# Semantic ID Generation — Part 3: Learned Quantization & Comparison

**Based on**: *Recommender Systems with Generative Retrieval* (Rajput et al., NeurIPS 2023 — TIGER),
*Autoregressive Image Generation using Residual Quantization* (Lee et al., CVPR 2022),
*Neural Discrete Representation Learning* (Van den Oord et al., NeurIPS 2017)

---

Parts 1–2 covered random IDs, hierarchical K-means, and product quantization.
This part introduces the state-of-the-art approach:

- **RQ-VAE** — learned residual quantization via autoencoder (§5)

We then compare all four approaches on the same dataset (§6) and summarise (§7).

We regenerate the dataset and re-run previous approaches (same seed) so all
SC values are available for the final comparison.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from torch import optim

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "mediumpurple"]

---
## 1. Toy Dataset

We generate $N = 200$ items from 5 Gaussian clusters in two variants:

- **2D** ($D=2$): for scatter-plot visualisation throughout the notebook
- **64D** ($D=64$): for the quantisation approaches where higher dimensionality is needed

Both share the same cluster assignments so results are directly comparable.

### Semantic coherence metric

To quantify how well an ID scheme preserves neighbourhood structure, we define:

$$\text{SC}(f) = \frac{1}{N}\sum_{i=1}^{N} \frac{\bigl|\{j : j \in \text{kNN}_k(i) \;\land\; \text{prefix}_1(f(i)) = \text{prefix}_1(f(j))\}\bigr|}{k}$$

where $f$ maps items to IDs and $\text{kNN}_k(i)$ are the $k$ nearest neighbours by
embedding distance. $\text{prefix}_1$ denotes the first token of the ID.
SC $\in [0, 1]$: higher means neighbours share ID prefixes more often.

In [ ]:
N_ITEMS = 200
N_CLUSTERS = 5
ITEMS_PER_CLUSTER = N_ITEMS // N_CLUSTERS

# Cluster centres
centers_2d = np.array([
    [-3.0, -3.0],
    [ 3.0, -3.0],
    [ 0.0,  3.0],
    [-3.0,  3.0],
    [ 3.0,  3.0],
])

# Generate 2D embeddings
embeddings_2d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 2) * 0.8 + centers_2d[idx]
    for idx in range(N_CLUSTERS)
])
labels = np.repeat(np.arange(N_CLUSTERS), ITEMS_PER_CLUSTER)

# Generate 64D embeddings with the same cluster structure
centers_64d = np.random.randn(N_CLUSTERS, 64) * 3.0
embeddings_64d = np.vstack([
    np.random.randn(ITEMS_PER_CLUSTER, 64) * 0.5 + centers_64d[idx]
    for idx in range(N_CLUSTERS)
])

print(f"2D embeddings : {embeddings_2d.shape}")
print(f"64D embeddings: {embeddings_64d.shape}")
print(f"Clusters      : {N_CLUSTERS}, {ITEMS_PER_CLUSTER} items each")

In [ ]:
def semantic_coherence(
    embeddings: np.ndarray,
    ids: np.ndarray,
    k: int = 10,
) -> float:
    """Fraction of k-NN that share the first token of their ID.

    Args:
        embeddings: (N, D) array of item embeddings.
        ids: (N,) or (N, L) array of IDs. First column used as prefix.
        k: number of nearest neighbours.

    Returns:
        SC in [0, 1].
    """
    ids = np.asarray(ids)
    if ids.ndim == 2:
        prefixes = ids[:, 0]
    else:
        prefixes = ids

    dists = cdist(embeddings, embeddings, metric="euclidean")
    np.fill_diagonal(dists, np.inf)
    n_items = len(embeddings)
    sc_sum = 0.0
    for idx in range(n_items):
        neighbours = np.argsort(dists[idx])[:k]
        matches = np.sum(prefixes[neighbours] == prefixes[idx])
        sc_sum += matches / k
    return sc_sum / n_items

In [ ]:
# Regenerate dataset and re-run previous approaches (same seed)

# --- Random IDs baseline ---
ids_random = np.arange(N_ITEMS)
np.random.shuffle(ids_random)
sc_random = semantic_coherence(embeddings_2d, ids_random)
sc_random_64d = semantic_coherence(embeddings_64d, np.random.permutation(N_ITEMS))


# --- Hierarchical K-Means (from Part 2) ---
def hierarchical_kmeans(
    embeddings: np.ndarray, K: int = 3, max_depth: int = 3,
) -> np.ndarray:
    """Assign hierarchical K-means IDs."""
    n_items = len(embeddings)
    ids = np.zeros((n_items, max_depth), dtype=np.int64)
    def _recurse(indices: np.ndarray, depth: int) -> None:
        if depth >= max_depth or len(indices) <= K:
            return
        km = KMeans(n_clusters=min(K, len(indices)), random_state=0, n_init=10)
        km.fit(embeddings[indices])
        for cluster_idx in range(km.n_clusters):
            child_mask = km.labels_ == cluster_idx
            child_indices = indices[child_mask]
            ids[child_indices, depth] = cluster_idx
            _recurse(child_indices, depth + 1)
    _recurse(np.arange(n_items), depth=0)
    return ids

hk_ids_2d = hierarchical_kmeans(embeddings_2d, K=5, max_depth=3)
hk_ids_64d = hierarchical_kmeans(embeddings_64d, K=5, max_depth=3)
sc_hk_2d = semantic_coherence(embeddings_2d, hk_ids_2d)
sc_hk_64d = semantic_coherence(embeddings_64d, hk_ids_64d)


# --- Product Quantization (from Part 2) ---
class ProductQuantizer:
    """Product Quantization: split embedding into M subspaces, K-means each."""
    def __init__(self, D: int, M: int, K: int) -> None:
        self.D, self.M, self.K = D, M, K
        self.sub_dim = D // M
        self.codebooks: list[np.ndarray] = []
    def fit(self, embeddings: np.ndarray) -> None:
        self.codebooks = []
        for m_idx in range(self.M):
            sub = embeddings[:, m_idx * self.sub_dim : (m_idx + 1) * self.sub_dim]
            km = KMeans(n_clusters=self.K, random_state=0, n_init=10).fit(sub)
            self.codebooks.append(km.cluster_centers_)
    def encode(self, embeddings: np.ndarray) -> np.ndarray:
        codes = np.zeros((len(embeddings), self.M), dtype=np.int64)
        for m_idx in range(self.M):
            sub = embeddings[:, m_idx * self.sub_dim : (m_idx + 1) * self.sub_dim]
            codes[:, m_idx] = np.argmin(cdist(sub, self.codebooks[m_idx]), axis=1)
        return codes

pq = ProductQuantizer(D=64, M=8, K=16)
pq.fit(embeddings_64d)
pq_codes = pq.encode(embeddings_64d)
sc_pq_64d = semantic_coherence(embeddings_64d, pq_codes)

pq_2d_full = ProductQuantizer(D=2, M=2, K=4)
pq_2d_full.fit(embeddings_2d)
sc_pq_2d = semantic_coherence(embeddings_2d, pq_2d_full.encode(embeddings_2d))

print("Previous approaches re-computed:")
print(f"  Random IDs SC       : {sc_random:.3f} (2D), {sc_random_64d:.3f} (64D)")
print(f"  Hier. K-Means SC    : {sc_hk_2d:.3f} (2D), {sc_hk_64d:.3f} (64D)")
print(f"  Product Quant. SC   : {sc_pq_2d:.3f} (2D), {sc_pq_64d:.3f} (64D)")

---
## 5. Approach 4: RQ-VAE (Residual Quantization VAE)

The state-of-the-art approach from TIGER (Rajput et al., NeurIPS 2023).
An autoencoder maps items to a latent space, then quantises using
**residual quantization (RQ)**: Level 1 quantises the latent vector;
Level 2 quantises the *residual* (error from Level 1); Level 3 quantises
the residual of the residual; and so on.

### Encoder-Decoder

$$\mathbf{z} = \text{Encoder}(\mathbf{e}_i), \qquad \hat{\mathbf{e}}_i = \text{Decoder}(\hat{\mathbf{z}})$$

### Residual Quantization

Initialise $\hat{\mathbf{z}}_0 = \mathbf{0}$. At each level $l = 1, \ldots, M$:

$$\mathbf{r}_{l-1} = \mathbf{z} - \hat{\mathbf{z}}_{l-1}$$

$$c_l = \arg\min_{k} \|\mathbf{r}_{l-1} - \mathbf{d}_{l,k}\|^2$$

$$\hat{\mathbf{z}}_l = \hat{\mathbf{z}}_{l-1} + \mathbf{d}_{l, c_l}$$

The item's semantic ID is $(c_1, c_2, \ldots, c_M)$ and the final quantised vector:

$$\hat{\mathbf{z}} = \sum_{l=1}^{M} \mathbf{d}_{l, c_l}$$

### Training Loss

$$\mathcal{L} = \underbrace{\|\mathbf{e}_i - \text{Decoder}(\hat{\mathbf{z}})\|^2}_{\text{reconstruction}}
+ \beta \underbrace{\sum_{l=1}^{M}\|\text{sg}[\mathbf{r}_{l-1}] - \mathbf{d}_{l,c_l}\|^2}_{\text{codebook loss}}
+ \beta \underbrace{\sum_{l=1}^{M}\|\mathbf{r}_{l-1} - \text{sg}[\mathbf{d}_{l,c_l}]\|^2}_{\text{commitment loss}}$$

where $\text{sg}[\cdot]$ is the stop-gradient operator. The **straight-through estimator (STE)**
passes gradients through the $\arg\min$:

$$\hat{\mathbf{z}}_{\text{STE}} = \mathbf{z} + \text{sg}[\hat{\mathbf{z}}_{\text{quantized}} - \mathbf{z}]$$

In [ ]:
class ResidualQuantizer(nn.Module):
    """Multi-level residual quantization with straight-through estimator."""

    def __init__(self, latent_dim: int, M: int, K: int) -> None:
        super().__init__()
        self.M = M
        self.K = K
        self.latent_dim = latent_dim
        self.codebooks = nn.ModuleList([
            nn.Embedding(K, latent_dim) for _ in range(M)
        ])
        # Initialise codebook entries from unit normal
        for cb in self.codebooks:
            nn.init.normal_(cb.weight, std=0.1)

    def forward(
        self, z: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Quantize z via residual quantization.

        Returns:
            codes: (N, M) long tensor of codebook indices.
            quantized: (N, latent_dim) quantized vector (with STE gradients).
            codebook_loss: scalar.
            commitment_loss: scalar.
        """
        batch_size = z.shape[0]
        codes = torch.zeros(batch_size, self.M, dtype=torch.long, device=z.device)
        quantized = torch.zeros_like(z)
        codebook_loss = torch.tensor(0.0, device=z.device)
        commitment_loss = torch.tensor(0.0, device=z.device)

        for level in range(self.M):
            residual = z - quantized.detach()
            # Find nearest codebook entry
            dists = torch.cdist(residual.unsqueeze(0), self.codebooks[level].weight.unsqueeze(0))
            dists = dists.squeeze(0)  # (N, K)
            codes[:, level] = dists.argmin(dim=-1)
            selected = self.codebooks[level](codes[:, level])  # (N, latent_dim)

            # Losses
            codebook_loss = codebook_loss + F.mse_loss(selected, residual.detach())
            commitment_loss = commitment_loss + F.mse_loss(residual, selected.detach())

            # Straight-through estimator
            quantized = quantized + residual + (selected - residual).detach()

        return codes, quantized, codebook_loss, commitment_loss

In [ ]:
class RQVAE(nn.Module):
    """Residual Quantization VAE: encoder → RQ → decoder."""

    def __init__(
        self, input_dim: int, latent_dim: int, hidden_dim: int, M: int, K: int,
    ) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.quantizer = ResidualQuantizer(latent_dim, M, K)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(
        self, x: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass: encode → quantize → decode.

        Returns:
            recon: (N, input_dim) reconstructed embedding.
            codes: (N, M) codebook indices.
            codebook_loss, commitment_loss: scalar losses.
            z: (N, latent_dim) encoder output (before quantization).
        """
        z = self.encoder(x)
        codes, quantized, cb_loss, cm_loss = self.quantizer(z)
        recon = self.decoder(quantized)
        return recon, codes, cb_loss, cm_loss, z

    def encode_to_ids(self, x: torch.Tensor) -> torch.Tensor:
        """Return just the semantic IDs (no gradients)."""
        with torch.no_grad():
            z = self.encoder(x)
            codes, _, _, _ = self.quantizer(z)
        return codes

In [ ]:
# Train RQ-VAE on 64D embeddings
rqvae = RQVAE(input_dim=64, latent_dim=32, hidden_dim=128, M=4, K=16)
optimizer = optim.Adam(rqvae.parameters(), lr=1e-3)
beta = 0.25

x_train = torch.tensor(embeddings_64d, dtype=torch.float32)
n_epochs = 500
history_recon = []
history_quant = []

for epoch_idx in range(n_epochs):
    optimizer.zero_grad()
    recon, codes, cb_loss, cm_loss, z_enc = rqvae(x_train)
    recon_loss = F.mse_loss(recon, x_train)
    loss = recon_loss + beta * (cb_loss + cm_loss)
    loss.backward()
    optimizer.step()

    if epoch_idx % 10 == 0:
        history_recon.append(recon_loss.item())
        history_quant.append((cb_loss + cm_loss).item())

print(f"Final reconstruction loss: {history_recon[-1]:.4f}")
print(f"Final quantization loss : {history_quant[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(0, n_epochs, 10), history_recon, color="steelblue")
axes[0].set_title("Reconstruction Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")

axes[1].plot(range(0, n_epochs, 10), history_quant, color="darkorange")
axes[1].set_title("Codebook + Commitment Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

plt.suptitle("RQ-VAE Training Curves", y=1.02)
plt.tight_layout()
plt.show()

### Exercise 4

Implement the core residual quantization forward pass. Given a latent vector
$\mathbf{z}$ and $M$ codebooks, iteratively quantise the residual at each level.
Return the sequence of codebook indices (the semantic ID) and the total
quantised vector.

**Hint**: at each level, compute `residual = z - quantized`, find the nearest
codebook entry via `torch.cdist`, and accumulate into `quantized`.

In [ ]:
def residual_quantize(
    z: torch.Tensor,
    codebooks: list[torch.Tensor],
    M: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Perform residual quantization (inference only, no STE).

    Args:
        z: (N, D) latent vectors.
        codebooks: list of M tensors, each (K, D).
        M: number of quantization levels.

    Returns:
        codes: (N, M) long tensor of codebook indices.
        quantized: (N, D) sum of selected codebook entries.
    """
    N, D = z.shape
    codes = torch.zeros(N, M, dtype=torch.long)
    quantized = torch.zeros_like(z)

    for level in range(M):
        residual = z - quantized
        # YOUR CODE HERE:
        # 1. Compute distances from residual to codebooks[level]  -> (N, K)
        # 2. Find nearest codebook entry index                    -> (N,)
        # 3. Look up the codebook vectors                         -> (N, D)
        # 4. Add them to quantized
        # Hint: dists = torch.cdist(residual, codebooks[level])
        pass

    return codes, quantized

In [ ]:
# Verify against the trained RQ-VAE model
rqvae.eval()
with torch.no_grad():
    test_z = rqvae.encoder(x_train[:10])
trained_codebooks = [cb.weight.detach() for cb in rqvae.quantizer.codebooks]

test_codes, test_quantized = residual_quantize(test_z, trained_codebooks, M=4)
ref_codes = rqvae.encode_to_ids(x_train[:10])
torch.testing.assert_close(test_codes, ref_codes)

# Verify reconstruction error decreases with each level
errors = []
partial = torch.zeros_like(test_z)
for lvl in range(4):
    partial = partial + trained_codebooks[lvl][test_codes[:, lvl]]
    errors.append(torch.mean((test_z - partial) ** 2).item())

assert all(
    errors[lvl] >= errors[lvl + 1] - 1e-6 for lvl in range(len(errors) - 1)
), "Reconstruction error must decrease (or stay equal) with each level"
print(f"Errors by level: {[f'{err:.4f}' for err in errors]}")
print("Exercise 4 passed!")

In [ ]:
# Reconstruction error across quantization levels
with torch.no_grad():
    z_all = rqvae.encoder(x_train)

level_errors = []
partial_all = torch.zeros_like(z_all)
for lvl in range(4):
    all_codes = rqvae.encode_to_ids(x_train)
    partial_all = partial_all + trained_codebooks[lvl][all_codes[:, lvl]]
    level_errors.append(torch.mean((z_all - partial_all) ** 2).item())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, 5), level_errors, color="seagreen", edgecolor="white")
ax.set_xticks(range(1, 5))
ax.set_xticklabels([f"Level {lvl}" for lvl in range(1, 5)])
ax.set_xlabel("Cumulative quantization levels")
ax.set_ylabel("Latent MSE")
ax.set_title("Each residual level captures finer detail")
plt.tight_layout()
plt.show()

In [ ]:
# Codebook usage histograms
all_codes = rqvae.encode_to_ids(x_train).numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for lvl in range(4):
    counts = np.bincount(all_codes[:, lvl], minlength=16)
    axes[lvl].bar(range(16), counts, color="crimson", edgecolor="white", alpha=0.8)
    axes[lvl].set_title(f"Level {lvl + 1}")
    axes[lvl].set_xlabel("Codebook entry")
    if lvl == 0:
        axes[lvl].set_ylabel("Count")

plt.suptitle("Codebook Usage per Level (uniform = healthy)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train a small RQ-VAE on 2D data for visualisation
rqvae_2d = RQVAE(input_dim=2, latent_dim=8, hidden_dim=32, M=3, K=8)
opt_2d = optim.Adam(rqvae_2d.parameters(), lr=1e-3)
x_2d = torch.tensor(embeddings_2d, dtype=torch.float32)

for epoch_idx in range(800):
    opt_2d.zero_grad()
    recon, codes, cb_loss, cm_loss, z_enc = rqvae_2d(x_2d)
    loss = F.mse_loss(recon, x_2d) + 0.25 * (cb_loss + cm_loss)
    loss.backward()
    opt_2d.step()

rqvae_2d.eval()
codes_2d = rqvae_2d.encode_to_ids(x_2d).numpy()

fig, ax = plt.subplots(figsize=(7, 5.5))
markers = ["o", "s", "^", "D", "v", "P", "*", "X"]
cmap = plt.cm.Set1

for c0 in np.unique(codes_2d[:, 0]):
    for c1 in np.unique(codes_2d[:, 1]):
        mask = (codes_2d[:, 0] == c0) & (codes_2d[:, 1] == c1)
        if mask.any():
            ax.scatter(
                embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                c=[cmap(c0 / 8)], marker=markers[c1 % len(markers)],
                alpha=0.7, s=45, edgecolors="white", linewidths=0.5,
            )

ax.set_title("RQ-VAE IDs on 2D (color=Level 1, shape=Level 2)")
ax.set_xlabel("$e_1$")
ax.set_ylabel("$e_2$")
plt.tight_layout()
plt.show()

**Pros**: End-to-end learnable; residual structure means early levels capture
coarse structure, later levels capture fine details (hierarchical by construction);
codebooks interact across levels (unlike PQ); state of the art on generative
retrieval benchmarks (TIGER).

**Cons**: Requires training a neural network; codebook collapse is a known failure
mode (mitigated by EMA updates or entropy regularisation); the number of levels $M$
and codebook size $K$ are hyperparameters; encoder/decoder add parameters and compute.

Once items have RQ-VAE semantic IDs, a Transformer decoder can be trained to predict
$c_1, c_2, \ldots, c_M$ autoregressively given a query embedding. The residual
structure means early tokens narrow to a region and later tokens refine within it —
naturally suited to beam search.

In [ ]:
# Final SC for RQ-VAE
rqvae_codes_64d = rqvae.encode_to_ids(x_train).numpy()
sc_rqvae_64d = semantic_coherence(embeddings_64d, rqvae_codes_64d)

sc_rqvae_2d = semantic_coherence(embeddings_2d, codes_2d)

print(f"{'Approach':<25} {'SC (2D)':>8} {'SC (64D)':>8}")
print(f"{'-' * 43}")
print(f"{'Random IDs':<25} {sc_random:>8.3f} {sc_random_64d:>8.3f}")
print(f"{'Hierarchical K-Means':<25} {sc_hk_2d:>8.3f} {sc_hk_64d:>8.3f}")
print(f"{'Product Quantization':<25} {sc_pq_2d:>8.3f} {sc_pq_64d:>8.3f}")
print(f"{'RQ-VAE':<25} {sc_rqvae_2d:>8.3f} {sc_rqvae_64d:>8.3f}")

---
## 6. Comprehensive Comparison

| Approach | Vocabulary Size | Learnable? | Reconstruction | Complexity |
|----------|----------------|------------|----------------|------------|
| Random/Atomic | $N$ | No | N/A | $O(1)$ |
| Hierarchical K-Means | $K \times L$ | No (offline) | Lossy | $O(NKL)$ |
| Product Quantization | $K \times M$ | No (offline) | Moderate MSE | $O(NMK)$ |
| RQ-VAE | $K \times M$ | Yes (end-to-end) | Low MSE | $O(\text{training})$ |

In [ ]:
approaches = ["Random", "Hier. K-Means", "PQ", "RQ-VAE"]
sc_values = [sc_random_64d, sc_hk_64d, sc_pq_64d, sc_rqvae_64d]
bar_colors = ["steelblue", "darkorange", "seagreen", "crimson"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(approaches, sc_values, color=bar_colors, edgecolor="white", width=0.6)
ax.set_ylabel("Semantic Coherence (SC)")
ax.set_title("64D Embeddings: SC across all four approaches")
ax.set_ylim(0, 1)
for idx, val in enumerate(sc_values):
    ax.text(idx, val + 0.02, f"{val:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ID similarity vs embedding distance for each approach
def compute_id_similarity(ids: np.ndarray) -> np.ndarray:
    """Shared prefix length between all pairs (normalised to [0, 1])."""
    if ids.ndim == 1:
        ids = ids.reshape(-1, 1)
    n_items = len(ids)
    n_levels = ids.shape[1]
    sim = np.zeros((n_items, n_items))
    for idx in range(n_items):
        for jdx in range(idx + 1, n_items):
            shared = 0
            for lvl in range(n_levels):
                if ids[idx, lvl] == ids[jdx, lvl]:
                    shared += 1
                else:
                    break
            sim[idx, jdx] = shared / n_levels
            sim[jdx, idx] = sim[idx, jdx]
    return sim


# Subsample for speed
n_sub = 80
sub_idx = np.random.choice(N_ITEMS, n_sub, replace=False)
sub_emb = embeddings_64d[sub_idx]
emb_dists = cdist(sub_emb, sub_emb).flatten()

all_ids_dict = {
    "Random": np.random.permutation(N_ITEMS)[sub_idx].reshape(-1, 1),
    "Hier. K-Means": hk_ids_64d[sub_idx],
    "PQ": pq_codes[sub_idx],
    "RQ-VAE": rqvae_codes_64d[sub_idx],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, (name, ids) in zip(axes, all_ids_dict.items()):
    id_sim = compute_id_similarity(ids).flatten()
    ax.scatter(emb_dists, id_sim, alpha=0.1, s=3, c="steelblue")
    ax.set_title(name)
    ax.set_xlabel("Embedding distance")
axes[0].set_ylabel("ID similarity\n(shared prefix)")

plt.suptitle("Closer embeddings should have more similar IDs", y=1.04)
plt.tight_layout()
plt.show()

### When to use what

- **Random IDs**: baseline only; never use for generative retrieval
- **Hierarchical K-Means**: simplicity and interpretability; corpus $< 1$M items;
  high-quality pre-computed embeddings; quick prototyping
- **Product Quantization**: fast ANN search; very large corpora;
  static embeddings; FAISS ecosystem
- **RQ-VAE**: when you can afford training; generative retrieval or recommendation;
  embeddings may be noisy or learned jointly; state-of-the-art performance needed

### References

1. Tay et al., "Transformer Memory as a Differentiable Search Index", NeurIPS 2022
2. Jégou et al., "Product Quantization for Nearest Neighbor Search", TPAMI 2011
3. Rajput et al., "Recommender Systems with Generative Retrieval", NeurIPS 2023 (TIGER)
4. Lee et al., "Autoregressive Image Generation using Residual Quantization", CVPR 2022 (RQ-VAE)
5. Van den Oord et al., "Neural Discrete Representation Learning", NeurIPS 2017 (VQ-VAE)

---
## 7. Summary

```
Core idea:
  Semantic IDs map items to structured tokens so that
  semantically similar items share ID prefixes or codes.

Four approaches (simplest → SOTA):
  1. Random IDs     — no structure; baseline
  2. Hierarchical K-means — tree path IDs via recursive clustering
  3. Product Quantization — independent subspace codebooks
  4. RQ-VAE         — end-to-end residual quantization with autoencoder

Key tradeoffs:
  - Vocabulary: N (atomic) vs K×L (hierarchical/PQ/RQ)
  - Learnability: offline clustering vs end-to-end gradient-based
  - Expressiveness: independent subspaces (PQ) vs interacting residuals (RQ)
  - Reconstruction: none (random) → lossy (K-means) → accurate (RQ-VAE)

For generative retrieval:
  RQ-VAE semantic IDs enable Transformer decoders to autoregressively
  generate item identifiers, with beam search naturally exploiting
  the coarse-to-fine structure of residual codes.
```

---
## Appendix: Exercise Solutions

Expand the cells below only after attempting the exercises.

In [ ]:
# Solution 4: residual_quantize
def residual_quantize_solution(
    z: torch.Tensor, codebooks: list[torch.Tensor], M: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    N, D = z.shape
    codes = torch.zeros(N, M, dtype=torch.long)
    quantized = torch.zeros_like(z)
    for level in range(M):
        residual = z - quantized
        dists = torch.cdist(residual, codebooks[level])
        codes[:, level] = dists.argmin(dim=-1)
        quantized = quantized + codebooks[level][codes[:, level]]
    return codes, quantized

print("Solution 4: residual = z - quantized; dists = cdist; codes = argmin; quantized += codebook[codes]")

---
## Further Exploration

1. **EMA codebook updates**: replace the commitment loss with exponential moving average
   updates to prevent codebook collapse (as in VQ-VAE-2)
2. **Gumbel-Softmax quantization**: replace the $\arg\min$ + STE with a differentiable
   Gumbel-Softmax relaxation (temperature-annealed)
3. **Train a Transformer decoder**: given query embeddings, autoregressively predict
   $c_1, c_2, \ldots, c_M$ — the full generative retrieval pipeline
4. **Locality-Sensitive Hashing (LSH)**: add as a fifth approach between random IDs
   and hierarchical K-means for comparison
5. **Joint training**: learn the item embeddings and RQ-VAE codebooks simultaneously
   rather than quantising pre-trained embeddings